# 护栏：用自定义工具实现人在环中

许多工作流处在“完全自动化”和“凡事询问人工”之间的空隙里。费用审批是典型例子：智能体可以自行处理清晰的情形，但它应当知道何时把模糊的情形上报给人工审核。分寸很重要——事事上报的智能体让人疲于应对，而从不上报的智能体则很危险。

本笔记本围绕两个**自定义工具**构建一个费用审批者：用于清晰情形的 `decide()`，以及用于模糊情形的 `escalate()`。两者都会经由你的应用往返一次，正是在这里，你要么记录结果（decide），要么把它提交给审核人员（escalate）。

## 自定义工具是什么

此前 Cookbook 一直使用内置的 `agent_toolset`（bash、read、write 等），它们全部在沙箱容器内运行。**自定义工具**不同：当智能体调用其中一个时，会话会暂停并发出一个 `agent.custom_tool_use` 事件，你的应用看到这次调用，运行你想要的任意代码，然后 POST 回一个 `user.custom_tool_result` 事件。会话带着该结果恢复，结果进入智能体的上下文。

这种形态适合两种情形：

1. **数据位于沙箱无法触及的地方。** 凡是在你自己的网络边界之后的东西，都通过往返由智能体回调你的应用。
2. **你想让人进入环中，或者在每次调用之前设置你自己的审计与审批层。** 这正是本笔记本所做的：`decide` 和 `escalate` 不只是抽象意义上的“工具”，它们是业务逻辑和人工审核人员从智能体接管的接缝处。

（其他扩展模式，即 MCP 工具集和 `resources=` 仓库挂载，分别在 operate 笔记本和 orchestrate 笔记本中讲解。）

本笔记本分为两部分。A 部分在本地流式处理会话事件，并在每次自定义工具请求到来时作出响应。这种方式在开发期间很方便，因为一切发生在同一个进程内，你能实时看到行为。B 部分简要指向生产环境的 webhook 模式，该模式在 operate 笔记本中有端到端的完整讲解。

测试数据位于 `example_data/gate/`，包含一个 `policy.yaml` 和十二张票据，覆盖策略的每一个分支。

In [ ]:
import json
import os
from collections import Counter
from pathlib import Path

from anthropic import Anthropic
from utilities import wait_for_idle_status

MODEL = os.environ.get("COOKBOOK_MODEL", "claude-sonnet-4-6")

client = Anthropic()
FIXTURE = Path("example_data") / "gate"

## 1. 上传策略和票据

In [ ]:
policy = client.beta.files.upload(
    file=("policy.yaml", (FIXTURE / "policy.yaml").read_bytes(), "text/yaml")
)
receipts = client.beta.files.upload(
    file=(
        "receipts.jsonl",
        (FIXTURE / "inbox" / "receipts.jsonl").read_bytes(),
        "application/jsonl",
    )
)

## 2. 定义带有两个自定义工具的智能体

自定义工具在同一个 `tools=` 数组中声明，与内置工具集并列，使用 `"type": "custom"` 并为输入提供 JSON 模式。每条声明告诉模型该工具的用途（`description`）、应当用什么参数调用它（`input_schema`）以及它的名称。智能体决定何时调用它们；你的代码决定它们被调用时做什么。

这里我们同时保留内置的 `agent_toolset_20260401` 处于启用状态，这样智能体可以直接读取策略文件和票据。`decide` 和 `escalate` 是两个自定义工具，使每一次决策都成为一次往返。

In [ ]:
agent = client.beta.agents.create(
    name="cookbook-gate",
    # Tells Anthropic which cookbook this agent came from. Safe to remove.
    metadata={"anthropic_cookbook": "claude-cookbooks/cma-gate-human-in-the-loop"},
    model=MODEL,
    system=(
        "You are an expense approver. Read each receipt in "
        "receipts.jsonl against the policy in policy.yaml and make "
        "exactly ONE tool call per receipt. Call decide(receipt_id, "
        "action, reason) for clear cases, or escalate(receipt_id, "
        "question) for ambiguous ones (near thresholds, unclear "
        "categories, suspicious notes). Once you've called decide "
        "or escalate for a given receipt, that receipt is finalized "
        "— do not call either tool for it again. After processing "
        "all receipts exactly once, stop."
    ),
    tools=[
        {
            "type": "agent_toolset_20260401",
            "default_config": {
                "enabled": True,
                "permission_policy": {"type": "always_allow"},
            },
        },
        {
            "type": "custom",
            "name": "decide",
            "description": "Record a final approve/reject for a clear-cut receipt.",
            "input_schema": {
                "type": "object",
                "properties": {
                    "receipt_id": {"type": "string"},
                    "action": {"type": "string", "enum": ["approve", "reject"]},
                    "reason": {"type": "string"},
                },
                "required": ["receipt_id", "action", "reason"],
            },
        },
        {
            "type": "custom",
            "name": "escalate",
            "description": "Surface an ambiguous receipt for human review.",
            "input_schema": {
                "type": "object",
                "properties": {
                    "receipt_id": {"type": "string"},
                    "question": {"type": "string"},
                },
                "required": ["receipt_id", "question"],
            },
        },
    ],
)

env = client.beta.environments.create(
    name="cookbook-gate-env",
    config={"type": "cloud", "networking": {"type": "limited"}},
)

session = client.beta.sessions.create(
    environment_id=env.id,
    agent={"type": "agent", "id": agent.id, "version": agent.version},
    resources=[
        {"type": "file", "file_id": policy.id, "mount_path": "policy.yaml"},
        {"type": "file", "file_id": receipts.id, "mount_path": "receipts.jsonl"},
    ],
    title="Expense gate",
)
print(f"session: {session.id}")

## A 部分：开发期间在本地流式处理

驱动自定义工具智能体最简单的方式，是流式读取会话事件并在每次工具请求到来时作出响应。`decide` 调用会被记录日志，`escalate` 调用会在本地获得一个模拟的人工决定。在生产环境中，你会把上报事项放入队列，让真正的审核人员稍后处理——这正是 operate 笔记本所覆盖的内容。

In [ ]:
def simulate_human_review(receipt_id: str, question: str) -> str:
    # Real implementation would show this in a UI and await input.
    # Here: reject anything the agent flags as suspicious.
    return "reject" if "suspicious" in question.lower() else "approve"


# The iterate notebook factored its streaming loop out into
# `stream_until_end_turn`, and most other notebooks just import it.
# This one doesn't, because every decision the agent makes is a
# custom tool call, which means the session keeps going idle with
# `stop_reason.type == "requires_action"` and
# `stop_reason.event_ids` pointing at the `agent.custom_tool_use`
# events waiting for a response. We POST a `user.custom_tool_result`
# for each, let the session resume, and eventually break on a
# `session.status_idle` that arrives with `end_turn`. The helper
# only knows how to exit on `end_turn`, so we need the full loop
# here.
decisions = {}  # receipt_id -> final decision record
tool_use_events = {}
responded_to = set()  # event_ids we've already replied to
print("=== Part A: streaming ===")
with client.beta.sessions.events.stream(session.id) as stream:
    client.beta.sessions.events.send(
        session_id=session.id,
        events=[
            {
                "type": "user.message",
                "content": [
                    {
                        "type": "text",
                        "text": (
                            "Read /mnt/session/uploads/policy.yaml and "
                            "/mnt/session/uploads/receipts.jsonl. Process "
                            "all 12 receipts. For each receipt, make "
                            "exactly one decide() or escalate() call and "
                            "then move on to the next. When every receipt "
                            "has been processed once, stop."
                        ),
                    }
                ],
            }
        ],
    )
    # Note on the responded_to set: when an agent emits more than 5
    # parallel custom tool calls, the server returns
    # `stop_reason.event_ids` as a sliding window of the next 5
    # pending. Each status_idle we observe in the stream has that
    # window pinned at the moment the event was emitted, but by the
    # time we iterate to the next status_idle event, the server has
    # already advanced past the events we just responded to. So we
    # need to dedupe across status_idle events to avoid double-
    # responding to the same custom tool call (which 400s).
    for ev in stream:
        if ev.type == "agent.custom_tool_use":
            tool_use_events[ev.id] = ev
        elif ev.type == "session.status_idle" and ev.stop_reason:
            if ev.stop_reason.type == "requires_action":
                for event_id in ev.stop_reason.event_ids:
                    if event_id in responded_to:
                        continue
                    tool_ev = tool_use_events[event_id]
                    name, args = tool_ev.name, tool_ev.input
                    receipt_id = args["receipt_id"]
                    if name == "decide":
                        decisions[receipt_id] = {"lane": args["action"], **args}
                        result = {"recorded": True}
                    elif name == "escalate":
                        human = simulate_human_review(receipt_id, args["question"])
                        decisions[receipt_id] = {
                            "lane": "escalated",
                            "human_decision": human,
                            **args,
                        }
                        result = {"human_decision": human}
                    else:
                        result = {"error": f"unknown tool {name}"}
                    client.beta.sessions.events.send(
                        session_id=session.id,
                        events=[
                            {
                                "type": "user.custom_tool_result",
                                "custom_tool_use_id": event_id,
                                "content": [{"type": "text", "text": json.dumps(result)}],
                            }
                        ],
                    )
                    responded_to.add(event_id)
            elif ev.stop_reason.type == "end_turn":
                break
        elif ev.type == "session.status_terminated":
            break

wait_for_idle_status(client, session.id)

lanes = Counter(d["lane"] for d in decisions.values())
print(f"\n{len(decisions)} decisions: {dict(lanes)}")

client.beta.sessions.archive(session.id)
client.beta.environments.archive(env.id)
client.beta.agents.archive(agent.id)
print("archived")

## B 部分：用于生产的 webhook

本地流式模式在开发期间表现良好，但它会在人工思考期间一直占用一个 HTTP 连接，这不利于扩展。生产模式改为在 Console 中注册一个 webhook，在 `session.status_idled` 时触发，该信号表示智能体要么已完成，要么正在等待工具结果。你的服务器检查事件，把任何待处理的上报提交给审核人员，并在人工完成时 POST 回 `user.custom_tool_result`，你这一侧不需要长连接。

operate 笔记本端到端讲解了完整的 webhook 配置：Console 注册、HMAC 签名校验、FastAPI 处理器，以及回到 `events.send` 的往返。响应智能体的代码与上面 A 部分完全相同；只有触发方式不同（webhook 推送取代流式拉取）。